In [16]:
from importlib.metadata import version
import tiktoken
print("Tiktoken Version:", version("tiktoken"))

Tiktoken Version: 0.14.0


In [17]:
tiktoken.list_encoding_names()

['gpt2',
 'r50k_base',
 'p50k_base',
 'p50k_edit',
 'cl100k_base',
 'o200k_base',
 'o200k_harmony']

In [18]:
tokenizer = tiktoken.get_encoding("gpt2")

In [19]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces "
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [20]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [21]:
with open("../data/combined/tolstoy.txt","r",encoding="utf-8") as f:
    raw_text = f.read()
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

2178265


In [22]:
enc_sample = enc_text[50000:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print(f"x: {x}")
print(f"y:        {y}")

x: [422, 683, 26, 290]
y:        [683, 26, 290, 4232]


In [23]:
for i in range(1,context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context),"------->",tokenizer.decode([desired]))

 from ------->  him
 from him -------> ;
 from him; ------->  and
 from him; and ------->  whatever


In [24]:
import torch
from torch.utils.data import Dataset,DataLoader

In [25]:
class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0,len(token_ids) - max_length,stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]

In [26]:
def create_dataloader_v1(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)
    dataloader = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)
    return dataloader

In [27]:
with open("../data/combined/tolstoy.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

In [40]:
max_length = 4
dataloader = create_dataloader_v1(raw_text,batch_size=8,max_length=max_length,stride=4,shuffle=False)
data_iter = iter(dataloader)
inputs,targets = next(data_iter)
print("Inputs:\n",inputs)
print("\nTargets:\n",targets)

Inputs:
 tensor([[ 3537,  4535,   509,  1503],
        [ 1677, 28893,   220,   628],
        [  416, 19632, 20054,   301],
        [  726,   220,   198, 30709],
        [16329,   198,   198, 14126],
        [  352,   198,   198, 25082],
        [ 4172,   389,   477, 12936],
        [   26,   790, 19283,  1641]])

Targets:
 tensor([[ 4535,   509,  1503,  1677],
        [28893,   220,   628,   416],
        [19632, 20054,   301,   726],
        [  220,   198, 30709, 16329],
        [  198,   198, 14126,   352],
        [  198,   198, 25082,  4172],
        [  389,   477, 12936,    26],
        [  790, 19283,  1641,   318]])


In [41]:
vocab_size = 50257
output_dim = 256
token_embedding_layer  = torch.nn.Embedding(vocab_size,output_dim)

In [42]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [43]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length,output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [44]:
input_embeddings = token_embeddings + pos_embeddings
print(pos_embeddings.shape)

torch.Size([4, 256])
